# Week 3: Pandas and `groupby`

This week we move from small Python values and lists to **real tabular data**.
We will use US baby-name records aggregated into the four Census regions:
Northeast, Midwest, South, and West.

By the end, you should be able to:

- recognise the main parts of a pandas DataFrame;
- select useful rows and columns;
- explain the difference between a Series and a one-column DataFrame;
- use `groupby` to answer questions about subsets of data; and
- check that convincing-looking output actually answers the question asked.




## Load pandas and the data

`pandas` is conventionally imported as `pd`. The abbreviation saves typing and
makes pandas code immediately recognisable.

`read_csv()` takes data stored in a CSV file and returns a DataFrame.

Throughout these notebooks, variables containing DataFrames end in `_df`, such
as `names_df`. Python does not require this. It is a small label for humans: when
you meet the variable later, its name reminds you what kind of object it contains.
We will use `_series` in the same way for important Series variables.

In [ ]:
import pandas as pd
from IPython.display import display

names_df = pd.read_csv("us_babynames_by_region.csv")
names_df.head()

Each row records how many babies of one sex received one name, in one Census
region and year. These counts were made by combining the US state files.

Very rare names are suppressed in the source state data for privacy, so
`Count` means **recorded count in this dataset**, not literally every birth.
This mainly matters when studying rare names.

## A DataFrame is a labelled table

A DataFrame is rather like a spreadsheet with stricter habits. It has:

- **rows**, containing observations;
- **columns**, containing variables;
- **column names**, describing those variables; and
- an **index**, containing labels pandas uses to identify rows.

![Anatomy of pandas data structures](pandas_illustration.jpeg)

In [ ]:
names_df.head()

The bold numbers on the left are the index. They are labels supplied by pandas;
they are not an extra baby-name variable.

A few useful ways to inspect an unfamiliar DataFrame are shown below.

In [ ]:
names_df.shape

In [ ]:
names_df.columns

In [ ]:
names_df.dtypes

`dtypes` tells us the kind of data stored in each column. `Year` and `Count`
are integers. Text columns may be displayed as `object` in pandas 2 or `str`
in pandas 3. The label can differ between versions; the important question is
what kind of values the column contains.

## One pair of brackets or two?

This tiny difference causes a surprising number of pandas puzzles:

```python
names_df["Name"]     # one column label -> Series
names_df[["Name"]]   # a list of column labels -> DataFrame
```

A **Series** is one labelled sequence. A **DataFrame** is a table.

A useful mental model is:

- a single label asks pandas to hand over **the column itself**;
- a list asks pandas to build **a smaller table using these columns**.

The list may contain only one name, but it is still a list—so pandas returns a
one-column table. The extra brackets are not decorative punctuation.

In [ ]:
name_series = names_df["Name"]
name_df = names_df[["Name"]]

display(name_series.head())
display(name_df.head())

print("Single label:", type(name_series), "shape:", name_series.shape)
print("List of labels:", type(name_df), "shape:", name_df.shape)

Notice that a Series shape has one number, such as `(1628156,)`, while a
one-column DataFrame shape has two, such as `(1628156, 1)`.

Neither form is universally better. We choose deliberately:

- use `names_df["Count"]` when a method or calculation expects a Series;
- use `names_df[["Count"]]` when you want the result to remain a table.

## Selecting columns and rows

Pandas has multiple ways to select columns and rows.  The first you'll see often is just giving it one thing.

A list of labels can select several columns and return all rows. 

`df[['column1','column2']]`

A range of numbers, can give you a set of rows, and all columns

`df[ 0:10 ]`

This usually works, but it works on some assumptions (rows are numbered, columns are labeled).  Next if you want to clearly select both rows and colums you use the `.loc[rows,columns]`   

`df[0:10,['column1','column2']]`






In [ ]:
names_df[[ "Year", "Name",]]


In [ ]:
names_df[0:10]

In [ ]:
names_df.loc[0:10,["Region", "Year", "Name"]]

Selecting rows by numbers is useful.  But a more useful selection method is to the "boolean" method.   To use this, instead of a range, we use a series that has 1 value for each row either `True` meaning keep the row, or `False` meaning skip the row. 

In [ ]:
#Here I'm going to create a boolean series for every row with the name Mary
#I do that by takin gthe names_df and selecting the Name column, then checking if it is equal to "Mary"
select_mary = names_df["Name"] == "Mary"

display(select_mary)


In [ ]:
#THe boolean selection works like this for just selecting rows and keeping all columns. 
names_df[select_mary]

In [ ]:
#Or like this with both selecting rows and columns. 
names_df.loc[select_mary, ["Region", "Year", "Name"]]


We are not limited to a single selection we can build up a more complicated picking logic by combining selections together with rule.

Each named selection asks one clear question about every row. Combining them with
`&` creates a final Boolean Series that is `True` only where all three conditions
are satisfied.

Also note that you can get the opposite of what is selected by using the boolean "Not" operated or `~` in python. 

The code inside `.loc[...]` then has two inputs, separated by a comma:

1. `select_rows` chooses rows;
2. `["Name", "Count"]` chooses columns.

Building the pieces separately makes them easy to inspect, reuse, or debug.

In [ ]:
# Build each question separately. Every result is a Boolean Series.
select_west = names_df["Region"] == "West"
select_female = names_df["Sex"] == "F"
select_male = ~select_female # ~ means "not", so this is the same as names_df["Sex"] == "M"
select_year = names_df["Year"] == 2020

#Notice we have a series here: 
display(select_year.shape)
display(names_df.shape)

Notice the output above for the boolean selector has a series (it's missing a 2nd value), and it's the same length as the dataframe

In [ ]:
display(select_female)

In [ ]:
# Combine the questions, then use the result to select rows.
#using & means "and", or all conditions must be true for the row to be selected.
select_rows = select_west & select_female & select_year
selected_df = names_df.loc[select_rows, ["Name", "Count"]]


selected_df.head()

### Live coding extension: add a threshold

The example above selected a place, sex, and year. Extend it by asking one more
question: which of those names were given to at least 30,000 babies in a region in a year?   Look at it for males and females. 



In [ ]:
# LIVE CODING EXTENSION
# 1. Create select_large_count using Count >= 30000.
# 2. Combine it with select_female, and select_year.


## Why we need `groupby`

Selection works nicely when we want one particular year or region. But suppose
we want an answer for **every** region, or every year.

Repeating nearly identical selections four or a hundred times would be tedious
and error-prone. `groupby` lets us state the repeated operation once.

Think of it as sorting a pile of forms into labelled trays:

1. **split** the rows into trays;
2. perform the same **calculation** inside every tray;
3. **combine** the tray summaries into a new table.

![Groupby separates rows into groups and condenses each group](groupby_max.png)

## A first `groupby`

Let us calculate the total recorded count in each region during 2020.

In [ ]:
select_2020 = names_df["Year"] == 2020
names_2020_df = names_df.loc[select_2020]

grouped_by_region = names_2020_df.groupby("Region", as_index=False)
regional_totals_df = grouped_by_region[["Count"]].sum()

regional_totals_df

Read the code in small stages:

- `select_2020` asks which rows belong to 2020;
- `names_2020_df` stores the selected DataFrame;
- `grouped_by_region` sorts those rows into regional groups;
- `regional_totals_df` adds `Count` inside every group.

The temporary `grouped_by_region` object is not itself a DataFrame. It is pandas'
plan for organising the rows. Calling `sum()` carries out the calculation and
returns a new DataFrame.

`as_index=False` keeps `Region` as an ordinary column. This is typically the most convenient organzition for new students.




## Grouping by more than one variable

Passing a list to `groupby` creates one group for every observed combination.
Here, each tray represents a region-and-sex combination.

In [ ]:
grouped_by_region_and_sex = names_2020_df.groupby(["Region", "Sex"], as_index=False)
region_and_sex_df = grouped_by_region_and_sex[["Count"]].sum()

region_and_sex_df

## The same groups can answer different questions

Once the rows have been divided into groups, we can ask different questions of
the same groups:

- `sum()` adds the recorded babies;
- `nunique()` counts the different names;
- `max()` finds the largest individual count.

For now, we will calculate each answer separately. This keeps the relationship
between the selected column and the calculation visible.

In [ ]:
grouped_by_region_and_sex = names_2020_df.groupby(["Region", "Sex"], as_index=False)

recorded_babies_df = grouped_by_region_and_sex[["Count"]].sum()
different_names_df = grouped_by_region_and_sex[["Name"]].nunique()
largest_counts_df = grouped_by_region_and_sex[["Count"]].max()

display(recorded_babies_df)
display(different_names_df)
display(largest_counts_df)

Each calculation returns one row for every region-and-sex group, but the numbers
mean different things because we selected a different column and method.

`different_names_df` counts names visible in the state files. 

Keeping these as separate tables is deliberately repetitive. At this stage,
I'm using it for clarity to help your understanding. Which is more valuable than compressing several calculations into one expression.

In practice people allmost never do that.   They do the grouping and aggregating in a single step.   This is important when you're working with large datasets.  

In [ ]:
#Here is an example of a more concise way to geth the same results above. 
summary_df = names_2020_df.groupby(["Region", "Sex"], as_index=False).agg({"Count": ["sum", "max"], "Name": "nunique"})
summary_df

### Live coding extension: compare two distant years

Extend the simple groupby by asking:

> How many different names were recorded in each region in 1920 and 2025?

This combines two skills: use named Boolean selections to choose two years, then
use `groupby` to answer the repeated regional question.

In [ ]:
# LIVE CODING EXTENSION
# 1. Create select_1920 and select_2025.
# 2. Select the rows from names_df that are in either year.
# 3. Group by Year and Region.
# 4. Select Name and use nunique().

## Warning when grouped columns become an index

Without `as_index=False`, pandas places the grouping labels in the result's
index. That is not wrong, but it can surprise beginners.

In [ ]:
grouped_with_index = names_2020_df.groupby(["Region", "Sex"])
indexed_result_df = grouped_with_index[["Count"]].sum()

indexed_result_df

In [ ]:
ordinary_columns_df = indexed_result_df.reset_index()
ordinary_columns_df

`reset_index()` turns those index labels back into ordinary columns. You will
see this frequently in real pandas code.

## Beware the Frankenstein row

Consider this tempting code:

In [ ]:
grouped_by_region = names_2020_df.groupby("Region", as_index=False)
frankenstein_df = grouped_by_region.max()

frankenstein_df

It runs. It produces a tidy table. It is also very easy to misread.

`max()` was calculated separately for every column:

- the latest alphabetic value in `Name`;
- the maximum value in `Count`;
- the maximum value in each other column.

Those values need not come from the same original row. Pandas has assembled a row
from unrelated body parts: technically a result, scientifically a monster.

For reference 

If we want the **complete row** for the most frequently recorded name, we can use
operations we already know:

1. sort the complete DataFrame from largest to smallest `Count`;
2. group the sorted rows;
3. take the first row from every group with `head(1)`.

Because `head(1)` selects actual rows, the values remain together.

In [ ]:
sorted_names_df = names_2020_df.sort_values("Count", ascending=False)

grouped_sorted_names = sorted_names_df.groupby(["Region", "Sex"])

most_popular_df = grouped_sorted_names.head(1)

most_popular_df = most_popular_df.sort_values(["Region", "Sex"])

most_popular_df

This is a useful example of operations forming a small pipeline without being
crammed into one line. We can inspect `sorted_names_df`, the grouped object, or the
final table separately if the result surprises us.

If two names have exactly the same count, `head(1)` returns whichever tied row
appears first after sorting. Handling every tied winner is a separate, harder task.

In [ ]:
display(sorted_names_df)

In [ ]:
display(grouped_sorted_names)

In [ ]:
display(most_popular_df.sort_values(["Region", "Sex"]))

In [ ]:
topnames_df = names_2020_df.groupby(["Region", "Sex"]).agg("head", n=1)

display(topnames_df.sort_values(["Region", "Sex"]))


In [ ]:
bottomnames_df = names_2020_df.groupby(["Region", "Sex"]).agg("tail", n=1)

display(bottomnames_df.sort_values(["Region", "Sex"]))

## Key ideas

- A DataFrame is a labelled table containing rows and columns.
- `frame["column"]` returns a Series.
- `frame[["column"]]` returns a one-column DataFrame.
- Boolean expressions select rows; lists of labels select columns.
- `groupby` means split, calculate, and combine.
- Select the intended value column before aggregating.
- Aggregated column maxima do not necessarily belong to one original row.
- Sorting before grouping lets `head(1)` retrieve one complete leading row per group.


